# 05 - Lakebase: operational serving for the coordinator app

**Why Lakebase:** 12 coordinators open the worklist all day and log every call. That workload needs millisecond reads per patient plus **transactional writes**, which is an OLTP pattern. A SQL warehouse is a poor fit for it.

| Object | Type | Direction |
| --- | --- | --- |
| `lakeview_ml.worklist_app_feed` -> `worklist_app_feed_synced` | Lakebase **synced table** (Triggered, from Change Data Feed) | Lakehouse -> Postgres |
| `lakeview_ml.facility_access` -> `facility_access_synced` | Synced table | Lakehouse -> Postgres (the app enforces facility scope from it) |
| `care.outreach_actions` | Native Postgres table, created and written by the app | App writes -> registered in UC for analytics |

**Design note:** Unity Catalog does not allow row filters or column masks on the source of a synced table (the error is reproduced below). So the governed `coordinator_worklist` serves analysts and Genie, while the restricted `worklist_app_feed` (GRANT-only) feeds Lakebase, and the app applies the same `facility_access` entitlements.

In [0]:
%pip install -q --upgrade "databricks-sdk>=0.118.0" "psycopg[binary]>=3.1"
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import (Project, ProjectSpec, SyncedTable, SyncedTableSyncedTableSpec,
                                             SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy)
w = WorkspaceClient()
PROJECT, BRANCH = "projects/lakeview-health", "projects/lakeview-health/branches/production"

try:
    project = w.postgres.get_project(name=PROJECT)
except Exception:
    # one-time: Postgres 17, autoscaling with scale-to-zero; auto-creates the production branch and primary endpoint
    project = w.postgres.create_project(project=Project(spec=ProjectSpec(display_name="Lakeview Health Care Coordination", pg_version=17)),
                                        project_id="lakeview-health").wait()
endpoint = next(iter(w.postgres.list_endpoints(parent=BRANCH)))
print(project.name, "|", endpoint.name, "|", endpoint.status.hosts.host)

projects/lakeview-health | projects/lakeview-health/branches/production/endpoints/primary | ep-billowing-glitter-d2kw32fj.database.us-east-1.cloud.databricks.com


In [0]:
# Expected to fail: coordinator_worklist carries a row filter and column masks (notebook 04)
try:
    w.postgres.create_synced_table(
        synced_table=SyncedTable(spec=SyncedTableSyncedTableSpec(
            source_table_full_name="simo_show_case_catalog.lakeview_ml.coordinator_worklist", branch=BRANCH,
            primary_key_columns=["encounter_id"], scheduling_policy=SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.TRIGGERED,
            postgres_database="databricks_postgres", create_database_objects_if_missing=True)),
        synced_table_id="simo_show_case_catalog.lakeview_ml.coordinator_worklist_synced")
except Exception as e:
    print("As expected ->", str(e)[:200])

As expected -> Table simo_show_case_catalog.lakeview_ml.coordinator_worklist cannot have both row/column security and online materialized views.


In [0]:
import time
SYNCS = {
    "simo_show_case_catalog.lakeview_ml.worklist_app_feed_synced": ("simo_show_case_catalog.lakeview_ml.worklist_app_feed", ["encounter_id"]),
    "simo_show_case_catalog.lakeview_ml.facility_access_synced":    ("simo_show_case_catalog.lakeview_ml.facility_access", ["principal", "facility_id"]),
}
for target, (source, pk) in SYNCS.items():
    try:
        w.postgres.get_synced_table(name=f"synced_tables/{target}")
        print("exists:", target)
    except Exception:
        w.postgres.create_synced_table(
            synced_table=SyncedTable(spec=SyncedTableSyncedTableSpec(
                source_table_full_name=source, branch=BRANCH, primary_key_columns=pk,
                scheduling_policy=SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.TRIGGERED,
                postgres_database="databricks_postgres", create_database_objects_if_missing=True)),
            synced_table_id=target)
        print("created:", target)

# wait for the initial snapshot
for _ in range(60):
    states = {t: str(w.postgres.get_synced_table(name=f"synced_tables/{t}").status.detailed_state) for t in SYNCS}
    if all("ONLINE" in s or "FAILED" in s for s in states.values()):
        break
    time.sleep(20)
print(states)

created: simo_show_case_catalog.lakeview_ml.worklist_app_feed_synced
created: simo_show_case_catalog.lakeview_ml.facility_access_synced
{'simo_show_case_catalog.lakeview_ml.worklist_app_feed_synced': 'SyncedTableState.SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE', 'simo_show_case_catalog.lakeview_ml.facility_access_synced': 'SyncedTableState.SYNCED_TABLE_ONLINE_TRIGGERED_UPDATE'}


In [0]:
import psycopg
user = w.current_user.me().user_name
token = w.postgres.generate_database_credential(endpoint=endpoint.name).token
with psycopg.connect(host=endpoint.status.hosts.host, dbname="databricks_postgres", user=user, password=token, sslmode="require") as conn:
    conn.read_only = True
    with conn.cursor() as cur:
        cur.execute("SELECT table_schema, table_name FROM information_schema.tables WHERE table_name LIKE '%synced' ORDER BY 1, 2")
        tables = cur.fetchall(); print("Postgres tables:", tables)
        for schema, name in tables:
            cur.execute(f'SELECT count(*) FROM "{schema}"."{name}"'); print(f"{schema}.{name}: {cur.fetchone()[0]} rows in Lakebase")
        cur.execute('SELECT facility_id, risk_tier, count(*) FROM "lakeview_ml"."worklist_app_feed_synced" GROUP BY 1, 2 ORDER BY 1, 2')
        print(cur.fetchall())
print("UC source rows:", spark.table("simo_show_case_catalog.lakeview_ml.worklist_app_feed").count())

Postgres tables: [('lakeview_ml', 'facility_access_synced'), ('lakeview_ml', 'worklist_app_feed_synced')]
lakeview_ml.facility_access_synced: 1 rows in Lakebase
lakeview_ml.worklist_app_feed_synced: 254 rows in Lakebase
[('LVH-CENTRAL', 'High', 24), ('LVH-CENTRAL', 'Low', 48), ('LVH-CENTRAL', 'Medium', 24), ('LVH-NORTH', 'High', 21), ('LVH-NORTH', 'Low', 45), ('LVH-NORTH', 'Medium', 24), ('LVH-SOUTH', 'High', 26), ('LVH-SOUTH', 'Low', 30), ('LVH-SOUTH', 'Medium', 12)]
UC source rows: 254


In [0]:
# Run once after the app resource is added. Gives the app's service principal (its Postgres role = app client id)
# read-only access to the synced worklist and entitlement tables. The app creates and owns its own write schema `care`.
APP_SP = "10b47283-51c7-4924-bcb8-c1f979d5b51f"   # lakeview-care-worklist service_principal_client_id
token = w.postgres.generate_database_credential(endpoint=endpoint.name).token
with psycopg.connect(host=endpoint.status.hosts.host, dbname="databricks_postgres", user=w.current_user.me().user_name,
                     password=token, sslmode="require", autocommit=True) as conn:
    conn.execute(f'GRANT USAGE ON SCHEMA lakeview_ml TO "{APP_SP}"')
    conn.execute(f'GRANT SELECT ON lakeview_ml.worklist_app_feed_synced, lakeview_ml.facility_access_synced TO "{APP_SP}"')
    print(conn.execute("SELECT table_name, privilege_type FROM information_schema.role_table_grants WHERE grantee = %s", (APP_SP,)).fetchall())

[('facility_access_synced', 'SELECT'), ('worklist_app_feed_synced', 'SELECT')]
